# 🛡️ Elastic Detection Rules — TOML ➜ Kibana NDJSON Converter

**Automated pipeline to bulk-convert Elastic's official Detection Rules (`.toml`) into Kibana-importable `.ndjson` files.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
![Python](https://img.shields.io/badge/python-3.12%2B-blue)
![License](https://img.shields.io/badge/license-MIT-green)
![Maintained](https://img.shields.io/badge/maintained-yes-brightgreen)

> ⭐ If this notebook saved you time, please **star** the repo — it helps other security engineers find it too!


## 📑 Table of Contents

1. [Overview](#overview)
2. [How It Works](#how-it-works)
3. [Prerequisites](#prerequisites)
4. [Step 1 — Install Python Dependencies](#step-1)
5. [Step 2 — Clone the Elastic Detection Rules Repository](#step-2)
6. [Step 3 — Install Poetry](#step-3)
7. [Step 4 — Install Project Dependencies](#step-4)
8. [Step 5 — Verify the CLI](#step-5)
9. [Step 6 — Choose the Category](#step-6)
10. [Step 7 — Export Every Windows Rule as a Separate `.ndjson`](#step-7)
11. [Step 8 — Verify the Files](#step-8)
12. [Step 9 — Zip the Folder](#step-9)
13. [Step 10 — Download the ZIP](#step-10)
14. [Importing into Kibana](#importing)
15. [References](#references)
16. [License & Disclaimer](#license)


<a id="overview"></a>
## 📖 Overview

### What are Elastic Detection Rules?

[Elastic Detection Rules](https://github.com/elastic/detection-rules) is Elastic's official, open-source
repository of pre-built threat-detection content for the Elastic Security app (SIEM). Every rule (e.g. *"Suspicious
PowerShell Encoded Command"*, *"Potential Ransomware Behavior"*) is defined as a human-readable **`.toml`** file,
organized by platform (`windows`, `linux`, `macos`, `network`, `cloud`, etc.) under the `rules/` directory.

### Why convert TOML to NDJSON?

TOML is the *source-of-truth* format used **inside the GitHub repository** for version control, testing, and
collaboration. However, **Kibana's Security app does not import `.toml` files directly** — its "Import rules"
feature (Stack Management → Rules → Import) only accepts a **newline-delimited JSON (`.ndjson`)** file, which is
also the format Kibana produces when you *export* rules from the UI.

So, to load Elastic's community/prebuilt detection rules into your own Kibana instance (for example, in an
air-gapped environment, a staging cluster, or a lab where automatic rule updates aren't available), you first need
to **convert the `.toml` rule definitions into a single `.ndjson` bundle**.

### What this notebook does

This notebook automates that entire conversion pipeline end-to-end:

1. Clones the official `elastic/detection-rules` repository (always up to date with the latest rules).
2. Installs the repository's own Python CLI (`detection_rules`) via Poetry.
3. Uses the built-in `export-rules-from-repo` command to convert every `.toml` rule in a chosen category
   (e.g. all Windows rules) into one clean `.ndjson` file.
4. Packages the result into a downloadable `.zip` archive, ready to be imported into Kibana.

### Who is this for?

- Detection engineers who want a local, importable copy of Elastic's rule set.
- Teams running Kibana in restricted/offline environments without direct access to Elastic's rule-update service.
- Anyone building a **Detection-as-Code** pipeline who wants a repeatable, scriptable TOML → NDJSON step.


<a id="how-it-works"></a>
## ⚙️ How It Works

```
elastic/detection-rules (GitHub)
        │  git clone
        ▼
   rules/<platform>/*.toml   (source rule definitions)
        │  poetry run python -m detection_rules export-rules-from-repo
        ▼
   <output>.ndjson           (single, Kibana-importable bundle)
        │  Stack Management → Rules → Import rules
        ▼
   Rules loaded into Kibana Security app
```

The heavy lifting is done by Elastic's own `export-rules-from-repo` CLI command (shipped inside the
`detection-rules` repo itself), so this notebook is essentially a reproducible wrapper around Elastic's
officially supported tooling — not a custom parser. That means it stays correct even as Elastic's rule schema
evolves.


<a id="prerequisites"></a>
## ✅ Prerequisites

- Runs out-of-the-box on **Google Colab** (recommended) or any Linux/macOS machine with:
  - Python 3.12+ (check the target repo's `pyproject.toml` for the exact minimum version)
  - `git`
  - Internet access to `github.com` and `pypi.org`
- No Elastic Cloud / Kibana credentials are required for this notebook — it only performs a **local, offline
  file conversion**. You only need Kibana access later, when you actually import the resulting `.ndjson` file.


<a id="step-1"></a>
## 1️⃣ Install Python Dependencies

Installs lightweight helper libraries used for HTTP requests and TOML parsing. (The `detection_rules` CLI itself
brings its own dependencies via Poetry in Step 4 — this step just covers convenience utilities used elsewhere in
this notebook.)


In [ ]:
!pip install -q requests toml

<a id="step-2"></a>
## 2️⃣ Clone the Elastic Detection Rules Repository

This pulls down the **latest** version of Elastic's official rule set and CLI tooling directly from GitHub, so the
rules you export are always current as of when you run this notebook.


In [ ]:
!git clone --depth 1 https://github.com/elastic/detection-rules.git

<a id="step-3-cd"></a>
### Change into the Repository Directory

> **🔧 Correction:** the original notebook used a plain `cd detection-rules` command. In a Jupyter/Colab
> notebook, `cd` on its own is an *IPython automagic* that can behave inconsistently and is easy to shadow
> accidentally. The reliable way to change directory in a notebook — so that **every later cell** runs from the
> right working directory — is the explicit `%cd` magic command below.


In [ ]:
%cd detection-rules

<a id="step-3"></a>
## 3️⃣ Install Poetry

[Poetry](https://python-poetry.org/) is the dependency manager the `detection-rules` project uses to pin and
install its exact Python dependencies (schema validation libraries, Click for the CLI, etc.).


In [ ]:
!pip install -q poetry

<a id="step-4"></a>
## 4️⃣ Install Project Dependencies

Reads `pyproject.toml` / `poetry.lock` in the cloned repo and installs the exact dependency versions the
`detection_rules` CLI was built and tested against. This can take a couple of minutes the first time.


In [ ]:
!poetry install

<a id="step-5"></a>
## 5️⃣ Verify the CLI Installed Correctly

If this prints the `detection_rules` help/usage banner (including commands like `export-rules-from-repo` and
`kibana`), the environment is ready.


In [ ]:
!poetry run python -m detection_rules --help

<a id="step-6"></a>
## 6️⃣ Choose the Category

Elastic organizes rules by platform under `rules/` (e.g. `windows`, `linux`, `macos`, `network`, `cloud`). Listing
the available categories here makes it easy to see what else you can target — the steps below use `rules/windows`
as the running example.


In [ ]:
import os

RULES_ROOT = "rules"
print("Available rule categories:\n")
for entry in sorted(os.listdir(RULES_ROOT)):
    full_path = os.path.join(RULES_ROOT, entry)
    if os.path.isdir(full_path):
        print(f" - {entry}")


<a id="step-7"></a>
## 7️⃣ Export Every Windows Rule as a Separate `.ndjson`

Walks `rules/windows`, and for every `.toml` rule found, calls `export-rules-from-repo` with `-f` (the single
rule file) and `-o` (its own output path), producing one `.ndjson` per rule under `Windows_NDJSON/`.

> To target a different category from Step 6, change `RULES_DIR` and `OUTPUT_DIR` below — e.g.
> `RULES_DIR = "rules/linux"`, `OUTPUT_DIR = "Linux_NDJSON"`.

Run this entire Python cell:


In [ ]:
import os
import subprocess

RULES_DIR = "rules/windows"
OUTPUT_DIR = "Windows_NDJSON"

os.makedirs(OUTPUT_DIR, exist_ok=True)

success = 0
failed = 0

for root, _, files in os.walk(RULES_DIR):
    for file in files:
        if file.endswith(".toml"):

            rule_path = os.path.join(root, file)

            output_name = file.replace(".toml", ".ndjson")
            output_path = os.path.join(OUTPUT_DIR, output_name)

            print(f"Exporting {file}...")

            result = subprocess.run(
                [
                    "poetry",
                    "run",
                    "python",
                    "-m",
                    "detection_rules",
                    "export-rules-from-repo",
                    "-f",
                    rule_path,
                    "-o",
                    output_path,
                ],
                capture_output=True,
                text=True,
            )

            if result.returncode == 0:
                success += 1
            else:
                failed += 1
                print(f"Failed: {file}")
                print(result.stderr)

print()
print("=" * 60)
print(f"Exported : {success}")
print(f"Failed   : {failed}")
print("=" * 60)


<a id="step-8"></a>
## 8️⃣ Verify the Files

Lists a sample of the exported `.ndjson` files to confirm the export worked.

You should see something like:

```
Windows_NDJSON/
    collection_email_outlook_mailbox_via_com.ndjson
    credential_access_lsass_memory_dump.ndjson
    defense_evasion_amsi_bypass_powershell.ndjson
    ...
```


In [ ]:
!find Windows_NDJSON -name "*.ndjson" | head

<a id="step-9"></a>
## 9️⃣ Zip the Folder

Packages every exported `.ndjson` file into a single archive: `Windows_NDJSON.zip`.


In [ ]:
import shutil

shutil.make_archive("Windows_NDJSON", "zip", "Windows_NDJSON")

<a id="step-10"></a>
## 🔟 Download the ZIP

Downloads `Windows_NDJSON.zip` to your machine. This only works inside **Google Colab** — if you're running
locally, the zip is already sitting on disk at that same path.


In [ ]:
from google.colab import files

files.download("Windows_NDJSON.zip")

<a id="importing"></a>
## 📥 Importing the NDJSON into Kibana

1. Open **Kibana** → **Security** → **Manage** → **Rules** → **Detection rules (SIEM)**.
2. Click **Import rules** (top right of the rules table).
3. Drag and drop (or browse to) one or more `.ndjson` files from `Windows_NDJSON/` (produced in Step 7), or the
   unzipped contents of `Windows_NDJSON.zip` from Step 9.
4. Optionally enable:
   - **Overwrite existing detection rules with conflicting `rule_id`** — updates rules that already exist.
   - **Overwrite existing exception lists with conflicting `list_id`** — same idea, for exceptions.
5. Click **Import** and confirm the rule count matches what you expected.

> 💡 If a rule references a Kibana **data view**, only the `data_view_id` reference is exported — the destination
> Kibana instance needs a data view with a matching ID, or you'll need to reassign it manually after import.


<a id="references"></a>
## 📚 References

- Elastic Detection Rules repository: https://github.com/elastic/detection-rules
- CLI reference (`export-rules-from-repo`, `import-rules-to-repo`, `kibana` commands): https://github.com/elastic/detection-rules/blob/main/CLI.md
- Elastic docs — Manage detection rules (import/export): https://www.elastic.co/docs/solutions/security/detect-and-alert/manage-detection-rules
- Elastic Security Labs — Detection as Code capabilities: https://www.elastic.co/security-labs/dac-beta-release


## 🤝 Contributing

Found a bug, want to add support for exporting exceptions/action connectors, or add a scheduled GitHub Action that
re-runs this pipeline nightly? PRs and issues are welcome!

If this notebook was useful, please consider:

- ⭐ **Starring** the repository
- 🍴 **Forking** it to adapt for your own environment
- 🐛 Opening an issue if something breaks against a newer `detection-rules` release


<a id="license"></a>
## ⚖️ License & Disclaimer

This notebook is an independent automation wrapper and is **not officially affiliated with or endorsed by
Elastic**. It simply automates Elastic's own, publicly documented CLI commands from the
[`elastic/detection-rules`](https://github.com/elastic/detection-rules) repository, which is separately licensed
by Elastic (see that repository's `LICENSE.txt` for the terms governing the rules and CLI code itself).

Suggested license for **this** wrapper notebook/repository: MIT.
